## How Databricks Connects to Cloud Storage
```
┌─────────────────────────────────────────────┐
│         CLOUD STORAGE (you own it)          │
│    S3 bucket / ADLS container / GCS         │
│    e.g., s3://my-company-data/              │
└────────────────────┬────────────────────────┘
                     │
                     │  Databricks connects to
                     │  SPECIFIC PATHS (not everything)
                     ↓
┌─────────────────────────────────────────────┐
│        EXTERNAL LOCATION (UC object)        │
│    s3://my-company-data/sales/              │
│    s3://my-company-data/marketing/          │
│    (specific paths, not the whole bucket)   │
└────────────────────┬────────────────────────┘
                     │
                     │  Governed by Unity Catalog
                     ↓
┌─────────────────────────────────────────────┐
│     UNITY CATALOG (governance layer)        │
│    - Grants: who can read/write             │
│    - Lineage: where data flows              │
│    - Audit: who accessed what, when         │
└────────────────────┬────────────────────────┘
                     │
                     ↓
              USERS & JOBS
```

### Layer by Layer

| Layer | What It Is | Example |
|---|---|---|
| **Cloud Storage** | Your own bucket/container; you own and pay for it | `s3://my-company-data/` |
| **External Location** | A Unity Catalog object that exposes only specific paths | `s3://my-company-data/sales/` |
| **Unity Catalog** | Governance layer: grants, lineage, audit | `GRANT SELECT ON ...` |
| **Users & Jobs** | Notebooks, SQL, pipelines that read/write the data | Analysts, ETL jobs |

### Key Points

- Databricks never gets access to the whole bucket, only to the specific paths registered as External Locations.
- Every read and write goes through Unity Catalog, so permissions, lineage and audit logs apply automatically.
- Users never handle cloud credentials directly; Unity Catalog uses a **Storage Credential** behind the scenes.

##DBFS 
 `a distributed file system that lets you access cloud object storage using file-like paths (like /mnt/..., /FileStore/...) instead of cloud-specific URLs.`

```display(dbutils.fs.ls('s3://d1ccsg-sas-migration/AUTONATION/DATA/'))```

```spark.read.text('s3://d1ccsg-sas-migration/AUTONATION/DATA/Service Scorecard/Input/RDM_PARTS_FACT.txt').show(2, truncate=False)```

###Why the Path Works
```text
Unity Catalog does not let you access any random cloud path.

It only lets you access paths that have been formally registered as an External Location.

So this works because the External Location covers it. If you tried s3://some-other-bucket/... 
— it would fail (no external location, no governance).```

